# 01 · Desempate entre modelos-base

Compara **Gemma 4 E4B** e **Qwen3.5-4B**, sem nenhum treino, nas perguntas de `data/avaliacao/perguntas_desempate.csv`.

Cada pergunta roda em dois modos:

- **sem contexto**: só a pergunta. Mede o quanto o modelo sabe, e o quanto inventa.
- **com contexto**: a pergunta mais o trecho do documento. Mede se o modelo usa bem o trecho e se recusa quando o trecho não traz a resposta.

**Como usar**

1. Ative a GPU T4 (Colab: *Ambiente de execução › Alterar o tipo*; Kaggle: *Settings › Accelerator*, com *Internet* ligada).
2. Escolha o modelo na célula de configuração e execute tudo.
3. Reinicie o ambiente, troque o modelo e execute de novo. Dois modelos na mesma sessão não cabem com folga na T4.
4. Na última seção, junte os resultados numa planilha e dê as notas.

Os resultados ficam em `resultados/`, que o Colab apaga ao fechar a sessão. Baixe a planilha ou salve no Google Drive.

## 1. Instalação

In [ ]:
%%capture
!pip install --upgrade unsloth unsloth_zoo

## 2. Configuração

In [ ]:
# Escolha UM modelo por execução e reinicie o ambiente antes de trocar.
MODELOS = {
    # Versão já quantizada em 4 bits: evita falta de memória ao carregar na T4.
    "gemma4_e4b": "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
    # Confira o nome exato no Hugging Face antes de rodar.
    "qwen35_4b": "Qwen/Qwen3.5-4B",
}
MODELO = "gemma4_e4b"   # troque para "qwen35_4b" na segunda execução

MAX_NOVOS_TOKENS = 400
REPO_URL = "https://github.com/iamrodrigocoelho/refauna-slm"

## 3. Perguntas

In [ ]:
import os, subprocess
import pandas as pd

if not os.path.exists("refauna-slm"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)

perguntas = pd.read_csv("refauna-slm/data/avaliacao/perguntas_desempate.csv")
print(f"{len(perguntas)} perguntas")
perguntas.groupby("tipo").size()

## 4. Carregar o modelo em 4 bits

In [ ]:
import torch
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODELOS[MODELO],
    max_seq_length=4096,
    load_in_4bit=True,
)
FastModel.for_inference(model)

print(f"Memória de vídeo ocupada após carregar: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 5. Prompts

O prompt de sistema é o mesmo para os dois modelos. No modo com contexto, o modelo deve responder só com base no trecho e dizer que não sabe quando o trecho não basta.

In [ ]:
SISTEMA = (
    "Você é um assistente especializado em refaunação e conservação da Mata Atlântica. "
    "Responda sempre em português do Brasil, de forma curta e precisa."
)

SISTEMA_COM_CONTEXTO = SISTEMA + (
    " Use apenas o trecho fornecido. Se o trecho não contiver a resposta, "
    "diga claramente que não há informação suficiente e não invente."
)

def mensagens(pergunta, trecho=None):
    if trecho is None:
        sistema, usuario = SISTEMA, pergunta
    else:
        sistema = SISTEMA_COM_CONTEXTO
        usuario = f"Trecho:\n{trecho}\n\nPergunta: {pergunta}"
    return [
        {"role": "system", "content": [{"type": "text", "text": sistema}]},
        {"role": "user", "content": [{"type": "text", "text": usuario}]},
    ]

## 6. Gerar as respostas

In [ ]:
import re, time

def aplicar_template(msgs):
    kwargs = dict(add_generation_prompt=True, tokenize=True,
                  return_dict=True, return_tensors="pt")
    try:
        # Desliga o modo de raciocínio, quando o template do modelo aceita.
        return tokenizer.apply_chat_template(msgs, enable_thinking=False, **kwargs)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, **kwargs)

def gerar(msgs):
    entrada = aplicar_template(msgs).to("cuda")
    n_entrada = entrada["input_ids"].shape[1]
    inicio = time.time()
    with torch.no_grad():
        saida = model.generate(**entrada, max_new_tokens=MAX_NOVOS_TOKENS,
                               do_sample=False, use_cache=True)
    segundos = time.time() - inicio
    novos = saida[0][n_entrada:]
    texto = tokenizer.decode(novos, skip_special_tokens=True)
    texto = re.sub(r"<think>.*?</think>", "", texto, flags=re.S).strip()
    return texto, segundos, len(novos)

torch.cuda.reset_peak_memory_stats()
linhas = []
for _, p in perguntas.iterrows():
    for modo, trecho in [("sem_contexto", None), ("com_contexto", p["trecho"])]:
        resposta, segundos, tokens = gerar(mensagens(p["pergunta"], trecho))
        linhas.append({
            "id": p["id"], "tipo": p["tipo"], "modo": modo,
            "pergunta": p["pergunta"], "resposta_esperada": p["resposta_esperada"],
            "modelo": MODELO, "resposta": resposta,
            "segundos": round(segundos, 1), "tokens": tokens,
        })
        print(f"[{p['id']} · {modo}] {resposta[:150]}")

pico = torch.cuda.max_memory_allocated() / 1e9
print(f"\nPico de memória de vídeo: {pico:.1f} GB")

## 7. Salvar

In [ ]:
os.makedirs("resultados", exist_ok=True)
resultado = pd.DataFrame(linhas)
caminho = f"resultados/desempate_{MODELO}.csv"
resultado.to_csv(caminho, index=False)

resumo = resultado.groupby("modo")[["segundos", "tokens"]].mean().round(1)
resumo["tokens_por_segundo"] = (resumo["tokens"] / resumo["segundos"]).round(1)
print(f"Salvo em {caminho}")
resumo

## 8. Juntar os dois modelos e dar as notas

Rode esta seção depois de executar o notebook com os dois modelos. Se a sessão foi reiniciada, envie os dois CSVs de volta para a pasta `resultados/` antes.

A planilha gerada traz as respostas lado a lado. Dê uma nota por resposta:

- **2**: correta e fiel ao trecho (ou recusa correta numa pergunta fora do escopo);
- **1**: parcialmente correta, ou correta mas com algo inventado;
- **0**: errada, inventada, ou em outro idioma.

Critério de escolha: vence quem tiver a maior nota média **com contexto**, desde que não perca feio em recusa. Registre a decisão em `docs/diario.md`.

In [ ]:
import glob

arquivos = sorted(glob.glob("resultados/desempate_*.csv"))
print("Arquivos encontrados:", arquivos)
todos = pd.concat([pd.read_csv(a) for a in arquivos])

lado_a_lado = todos.pivot_table(
    index=["id", "tipo", "modo", "pergunta", "resposta_esperada"],
    columns="modelo", values="resposta", aggfunc="first",
).reset_index()
for m in todos["modelo"].unique():
    lado_a_lado[f"nota_{m}"] = ""

lado_a_lado.to_excel("resultados/desempate_para_notas.xlsx", index=False)
print("Planilha salva em resultados/desempate_para_notas.xlsx")
lado_a_lado.head()